In [1]:
# MT Adaptation with SmolDoc: SFT on H100
import os
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer, SFTConfig
from utils import get_extended_datasets

torch.set_float32_matmul_precision("high")

# --- CONFIG ---
SMOLDOC_CONFIG = "smoldoc__en_sw"
SFT_MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct" # Base model
OUTPUT_DIR_SFT = f"checkpoints/sft_{SMOLDOC_CONFIG}"
os.makedirs("checkpoints", exist_ok=True)

print("Config:", SMOLDOC_CONFIG)
print("SFT base model:", SFT_MODEL_NAME)

# --- Load Dataset ---
datasets = get_extended_datasets(save_path="data/smoldoc_datasets", overwrite=False)
ds_full: Dataset = datasets[SMOLDOC_CONFIG]

# Split: 90% train_eval, 10% test
split_1 = ds_full.train_test_split(test_size=0.1, seed=42)
train_eval_ds = split_1["train"]
test_ds = split_1["test"]

# Split train_eval: 80% train, 20% eval
split_2 = train_eval_ds.train_test_split(test_size=0.2, seed=42)
train_ds = split_2["train"]
eval_ds = split_2["test"]

print(f"Train size: {len(train_ds)}")
print(f"Eval size: {len(eval_ds)}")
print(f"Test size: {len(test_ds)}")

Config: smoldoc__en_sw
SFT base model: meta-llama/Meta-Llama-3-8B-Instruct
📂 Found existing SmolDoc DatasetDict at data/smoldoc_datasets, loading from disk...
📂 Loaded DatasetDict from data/smoldoc_datasets with 102 configs.
Using cached file: data/smoldoc-factuality-ratings.json
Train size: 420
Eval size: 105
Test size: 59


In [2]:
# --- Tokenizer & Chat Template ---
tokenizer_sft = AutoTokenizer.from_pretrained(SFT_MODEL_NAME)

llama3_chat_template = """{% set loop_messages = messages %}{% for message in loop_messages %}{% set content = '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n'+ message['content'] | trim + '<|eot_id|>' %}{% if loop.index0 == 0 %}{% set content = bos_token + content %}{% endif %}{{ content }}{% endfor %}{% if add_generation_prompt %}{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}{% endif %}"""


tokenizer_sft.chat_template = llama3_chat_template
if tokenizer_sft.pad_token is None:
    tokenizer_sft.pad_token = tokenizer_sft.eos_token
tokenizer_sft.padding_side = "right"

# --- Data Formatting (Conversational) ---

def format_mt_example_to_messages(srcs, trgs):
    """
    Converts inputs to the specific 'messages' format required by TRL.
    Structure: [{"role": "user", "content": ...}, {"role": "assistant", "content": ...}]
    """
    src = " ".join(srcs).strip()
    tgt = " ".join(trgs).strip()

    # Note: Gemma templates don't natively support a distinct 'system' role.
    # It is best practice to merge the system instruction into the first user message.
    system_instruction = "You are an expert in English to Swahili translation.\nTranslate the following English text into Swahili.\n\n"

    return [
        {"role": "user", "content": f"{system_instruction}English:\n{src}"},
        {"role": "assistant", "content": f"Swahili:\n{tgt}"}
    ]

def add_messages_column(batch):
    # Create the list of message objects
    messages_list = [
        format_mt_example_to_messages(srcs, trgs)
        for srcs, trgs in zip(batch["srcs"], batch["trgs"])
    ]
    return {"messages": messages_list}

# Apply mapping
# Apply mapping and remove original columns
train_ds_fmt = train_ds.map(
    add_messages_column,
    batched=True,
    remove_columns=train_ds.column_names  # Remove original columns
)
eval_ds_fmt = eval_ds.map(
    add_messages_column,
    batched=True,
    remove_columns=eval_ds.column_names
)
test_ds_fmt = test_ds.map(
    add_messages_column,
    batched=True,
    remove_columns=test_ds.column_names
)
print(f"\nAfter formatting:")
print(f"Train formatted size: {len(train_ds_fmt)}")
print(f"Eval formatted size: {len(eval_ds_fmt)}")
print(f"Train columns: {train_ds_fmt.column_names}")
# Preview
print("\nExample 'messages' format:")
print(train_ds_fmt[0]["messages"])

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]


After formatting:
Train formatted size: 420
Eval formatted size: 105
Train columns: ['messages']

Example 'messages' format:
[{'content': 'You are an expert in English to Swahili translation.\nTranslate the following English text into Swahili.\n\nEnglish:\nThe history of Cameroon is long and complex, dating back to the earliest human settlements in the region. The area was first settled by Bantu peoples around 3000 BCE, and it was later conquered by the Kanem-Bornu Empire in the 11th century. In the 15th century, the Portuguese arrived in Cameroon and established trade relations with the local people. The region was later colonized by Germany in the 1880s, and it became a German colony known as Kamerun. During World War I, Cameroon was occupied by British and French forces, and it was divided between the two countries after the war. The British-controlled region became known as Southern Cameroons, while the French-controlled region became known as French Cameroon. In 1960, Southern Ca

In [3]:
# --- SFT Configuration ---
# H100 Settings
per_device_bs = 4        # Increased for H100 (adjust if OOM, but 4B should fit easily)
grad_accum = 2          # Effective batch = 8 * 2 = 16

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR_SFT,
    num_train_epochs=3,
    per_device_train_batch_size=per_device_bs,
    per_device_eval_batch_size=per_device_bs,
    gradient_accumulation_steps=grad_accum,
    learning_rate=5e-5,
    max_length=1024,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    bf16=True,                       # Essential for H100
    packing=False,                   # TRL handles packing for messages differently, easier to keep False for now
    dataset_text_field="messages",   # TRL looks for this column specifically
    gradient_checkpointing=True,     # Recommended even on H100 for batch size efficiency
    optim="adamw_torch_fused",
)

# --- Initialize Model ---
print("\nLoading model...")
model_sft = AutoModelForCausalLM.from_pretrained(
    SFT_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)

# --- Create Trainer ---
# NOTE: formatting_func is REMOVED.
# TRL automatically detects the "messages" column and uses tokenizer.apply_chat_template
trainer_sft = SFTTrainer(
    model=model_sft,
    processing_class=tokenizer_sft, # 'tokenizer' argument is deprecated in newer TRL, use processing_class
    train_dataset=train_ds_fmt,
    eval_dataset=eval_ds_fmt,
    args=sft_config,
)

# --- Train ---
print("\n" + "="*50)
print("STARTING TRAINING")
print("="*50 + "\n")

trainer_sft.train()

# --- Save ---
final_dir = os.path.join(OUTPUT_DIR_SFT, "llama")
trainer_sft.save_model(final_dir)
tokenizer_sft.save_pretrained(final_dir)
print(f"\nTraining finished. Model saved to: {final_dir}")


Loading model...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

Truncating eval dataset:   0%|          | 0/105 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128009}.



STARTING TRAINING



Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,1.236900,1.121525,1.126179,264998.000000,0.728271
100,0.560800,0.935691,0.731318,524535.000000,0.771748
150,0.138300,1.044392,0.489575,790505.000000,0.773474



Training finished. Model saved to: checkpoints/sft_smoldoc__en_sw/llama


In [4]:
# --- Clean up training artifacts ---
import gc

print("Cleaning up training memory...")

# Delete trainer (this holds optimizer states, model, etc.)
del trainer_sft

# Delete the training model
del model_sft

# Delete datasets if you don't need them
del train_ds_fmt, eval_ds_fmt

# Force garbage collection
gc.collect()

# Clear CUDA cache
torch.cuda.empty_cache()

print("Memory cleaned!")
print(f"GPU memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")

Cleaning up training memory...
Memory cleaned!
GPU memory allocated: 0.06 GB
GPU memory reserved: 0.11 GB


In [5]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import evaluate
import os

# --- CONFIG FOR INFERENCE ---
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR_SFT, "llama")

print("\n\n" + "="*50)
print("LOADING MODEL FOR GENERATION-BASED EVALUATION")
print("="*50)

# --- Load Model and Tokenizer for Inference ---
ft_tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)
if ft_tokenizer.pad_token is None:
    ft_tokenizer.pad_token = ft_tokenizer.eos_token
ft_tokenizer.padding_side = "left"  # standard for generation

ft_model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT_DIR,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",   # single H100
    attn_implementation="flash_attention_2",
)

pipe = pipeline(
    "text-generation",
    model=ft_model,
    tokenizer=ft_tokenizer,
    device_map="cuda:0",
)

print("\n\n" + "="*50)
print("FINAL TEST SET TRANSLATIONS (Held-out 10% Split)")
print("="*50)

# --- Define system instruction (must match training) ---
system_instruction = (
    "You are an expert in English to Swahili translation.\n"
    "Translate the following English text into Swahili.\n\n"
)

# --- Add prompt + reference columns to the dataset ---
def build_prompt(example):
    src_text = " ".join(example["srcs"]).strip()
    messages = [
        {"role": "user", "content": f"{system_instruction}English:\n{src_text}"}
    ]
    prompt = ft_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    example["prompt"] = prompt
    example["reference"] = " ".join(example["trgs"]).strip()
    return example

# map is non-batched by default -> our function expects single example dict
test_dataset = test_ds.map(build_prompt)

# --- Prepare lists for the pipeline (avoid Column object) ---
prompts = [p for p in test_dataset["prompt"]]       # force plain Python list
refs    = [r for r in test_dataset["reference"]]    # same for references

# --- Run batched inference ---
batch_size = 16  # can try 32/64 on H100 if VRAM is fine

# In the inference section, update your pipeline call:
outputs = pipe(
    prompts,
    max_new_tokens=1024,
    do_sample=False,
    return_full_text=False,
    pad_token_id=ft_tokenizer.eos_token_id,
    eos_token_id=ft_tokenizer.eos_token_id,  # Add this
    batch_size=batch_size,
)

# --- Extract predictions & references ---
all_predictions = []
all_references = []

for i, (out, ref_text) in enumerate(zip(outputs, refs)):
    generated_text = out[0]["generated_text"].strip()

    # Remove template artifacts and stop at first end token
    clean_prediction = generated_text.replace("Swahili:", "").strip()
    clean_prediction = clean_prediction.split("<|eot_id|>")[0].strip()  # Llama 3
    clean_prediction = clean_prediction.split("<end_of_turn>")[0].strip()  # If any Gemma tokens remain

    print(f"\n--- Example {i+1} / {len(test_ds)} ---")
    print(f"SOURCE (EN): {prompts[i]}")
    print(f"REFERENCE (SW): {ref_text}")
    print(f"MODEL OUTPUT (SW): {clean_prediction}")
    print("-"*50)

    all_predictions.append(clean_prediction)
    all_references.append([ref_text])


# --- Compute BLEU ---
bleu_metric = evaluate.load("bleu")
bleu_result = bleu_metric.compute(
    predictions=all_predictions,
    references=all_references,
)

print("\n\n" + "="*50)
print("TEST SET BLEU SCORE")
print("="*50)
print(f"BLEU: {bleu_result['bleu']:.4f}")
print(f"BLEU Precisions: {bleu_result['precisions']}")
print(f"Brevity Penalty: {bleu_result['brevity_penalty']:.4f}")
print(f"Length Ratio: {bleu_result['length_ratio']:.4f}")



LOADING MODEL FOR GENERATION-BASED EVALUATION


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0




FINAL TEST SET TRANSLATIONS (Held-out 10% Split)


Map:   0%|          | 0/59 [00:00<?, ? examples/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- Example 1 / 59 ---
SOURCE (EN): <|begin_of_text|><|start_header_id|>user<|end_header_id|>

You are an expert in English to Swahili translation.
Translate the following English text into Swahili.

English:
Patrick loved to spend his free time pursuing his hobbies. He was an avid reader and would often spend hours in the library or curled up in his favorite chair with a good book. He also enjoyed playing the guitar and would often practice for hours on end. Patrick also loved to go hiking and camping with his friends and family. He found peace and tranquility in the great outdoors.<|eot_id|><|start_header_id|>assistant<|end_header_id|>


REFERENCE (SW): Patrick alipenda kutumia muda wake huru kuendeleza mambo anayoyapenda. Alikuwa msomaji hodari na kila mara angetumia saa nyingi katika maktaba au kujikunja kwenye kiti chake akipendacho akiwa na kitabu kizuri. Pia alifurahia kucheza gitaa na kila mara angefanya mazoezi kwa saa nyingi. Pia Patrick alipenda kutembea kwa miguu na kupiga